In [1]:
# 1. imports and repo root

from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path
from typing import Any

import pandas as pd
import yaml


def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "configs").exists() and (candidate / "scripts").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing configs/, scripts/, and src/.")


REPO_ROOT = find_repo_root()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print("REPO_ROOT =", REPO_ROOT)

REPO_ROOT = C:\Users\quantbase\Desktop\fineas


In [2]:
# 2. load config sanity

run_spec_path = REPO_ROOT / "configs" / "runs" / "template_run_spec.json"
ingestion_cfg_path = REPO_ROOT / "configs" / "ingestion.yml"
universe_path = REPO_ROOT / "configs" / "universe.yml"

with run_spec_path.open("r", encoding="utf-8") as f:
    run_spec = json.load(f)

with ingestion_cfg_path.open("r", encoding="utf-8") as f:
    ingestion_cfg = yaml.safe_load(f)

with universe_path.open("r", encoding="utf-8") as f:
    universe_cfg = yaml.safe_load(f)


def flatten_symbols(obj: Any) -> list[str]:
    out: list[str] = []
    if isinstance(obj, dict):
        for v in obj.values():
            out.extend(flatten_symbols(v))
    elif isinstance(obj, list):
        for v in obj:
            out.extend(flatten_symbols(v))
    elif isinstance(obj, str):
        out.append(obj)
    return out


basket_name = run_spec["universe"]
basket_symbols = flatten_symbols(universe_cfg["baskets"][basket_name])

print("run_spec =", run_spec)
print("ingestion_cfg =", ingestion_cfg)
print("basket_name =", basket_name)
print("basket_size =", len(basket_symbols))
print("first_10_symbols =", basket_symbols[:10])

run_spec = {'run_id': 'run=20260309T000000Z', 'universe': 'demo_small', 'interval': '1d', 'start': '2018-01-01', 'end': '2026-03-10', 'features_enabled': ['returns', 'movement'], 'rag': {'window_size': 5, 'top_k': 5}}
ingestion_cfg = {'provider': 'yfinance', 'interval': '1d', 'start': '2018-01-01', 'end': '2026-03-10', 'auto_adjust': False, 'actions': True}
basket_name = demo_small
basket_size = 28
first_10_symbols = ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'GOOGL', 'META', 'BRK-B', 'JPM', 'V', 'WMT']


In [3]:
# helpers to run scripts and read reports

RUN_ID = run_spec["run_id"]
RUN_DIR = REPO_ROOT / "data" / "meta" / "runs" / RUN_ID


def run_script(script_relpath: str) -> subprocess.CompletedProcess:
    cmd = [sys.executable, str(REPO_ROOT / script_relpath)]
    print("RUNNING:", " ".join(cmd))
    proc = subprocess.run(
        cmd,
        cwd=str(REPO_ROOT),
        text=True,
        capture_output=True,
    )
    print("returncode =", proc.returncode)
    if proc.stdout:
        print("\n--- STDOUT (tail) ---")
        print("\n".join(proc.stdout.splitlines()[-40:]))
    if proc.stderr:
        print("\n--- STDERR (tail) ---")
        print("\n".join(proc.stderr.splitlines()[-40:]))
    return proc


def load_json_if_exists(path: Path) -> dict[str, Any] | None:
    if not path.exists():
        print("MISSING:", path)
        return None
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def print_report(name: str) -> dict[str, Any] | None:
    path = RUN_DIR / name
    report = load_json_if_exists(path)
    print(f"\n{name} ->", "FOUND" if report is not None else "MISSING")
    if report is not None:
        print(json.dumps(report, indent=2)[:4000])
    return report


print("RUN_DIR =", RUN_DIR)

RUN_DIR = C:\Users\quantbase\Desktop\fineas\data\meta\runs\run=20260309T000000Z


In [4]:
# run Phase 1 ingest

proc_ingest = run_script("scripts/01_ingest_yf.py")
ingest_report = print_report("ingest_report.json")

RUNNING: c:\Users\quantbase\.conda\envs\fineas\python.exe C:\Users\quantbase\Desktop\fineas\scripts\01_ingest_yf.py
returncode = 0

ingest_report.json -> FOUND
{
  "run_id": "run=20260309T000000Z",
  "created_utc": "2026-03-10T04:08:45.279810+00:00",
  "run_spec": {
    "universe": "demo_small",
    "interval": "1d",
    "start": "2018-01-01",
    "end": "2026-03-10",
    "features_enabled": [
      "returns",
      "movement"
    ],
    "rag": {
      "window_size": 5,
      "top_k": 5
    }
  },
  "ingestion_cfg": {
    "provider": "yfinance",
    "auto_adjust": false,
    "actions": true
  },
  "symbols": [
    "AAPL",
    "MSFT",
    "NVDA",
    "AMZN",
    "GOOGL",
    "META",
    "BRK-B",
    "JPM",
    "V",
    "WMT",
    "AMD",
    "CRM",
    "UBER",
    "CRWD",
    "SNOW",
    "SOFI",
    "BRK-A",
    "CELH",
    "^DJI",
    "^IXIC",
    "BTC-USD",
    "ETH-USD",
    "SOL-USD",
    "BNB-USD",
    "XRP-USD",
    "ADA-USD",
    "DOGE-USD",
    "LINK-USD"
  ],
  "results": {
    

In [5]:
# run Phase 2 normalization

proc_norm = run_script("scripts/02_build_norm.py")
norm_report = print_report("norm_report.json")

RUNNING: c:\Users\quantbase\.conda\envs\fineas\python.exe C:\Users\quantbase\Desktop\fineas\scripts\02_build_norm.py
returncode = 0

--- STDOUT (tail) ---
[fineas] wrote: C:\Users\quantbase\Desktop\fineas\data\meta\runs\run=20260309T000000Z\norm_report.json
[fineas] summary: ok_parts=2644/2772, qc_failed=0, error=0

norm_report.json -> FOUND
{
  "summary": {
    "run_id": "run=20260309T000000Z",
    "created_utc": "2026-03-10T04:14:27.398682+00:00",
    "interval": "1d",
    "start": "2018-01-01",
    "end": "2026-03-10",
    "symbols": [
      "AAPL",
      "MSFT",
      "NVDA",
      "AMZN",
      "GOOGL",
      "META",
      "BRK-B",
      "JPM",
      "V",
      "WMT",
      "AMD",
      "CRM",
      "UBER",
      "CRWD",
      "SNOW",
      "SOFI",
      "BRK-A",
      "CELH",
      "^DJI",
      "^IXIC",
      "BTC-USD",
      "ETH-USD",
      "SOL-USD",
      "BNB-USD",
      "XRP-USD",
      "ADA-USD",
      "DOGE-USD",
      "LINK-USD"
    ],
    "months": [
      [
        20

In [6]:
# run Phase 3 technicals

proc_feat = run_script("scripts/03_build_features.py")
features_report = print_report("features_report.json")

RUNNING: c:\Users\quantbase\.conda\envs\fineas\python.exe C:\Users\quantbase\Desktop\fineas\scripts\03_build_features.py
returncode = 0

--- STDOUT (tail) ---
[fineas] wrote: C:\Users\quantbase\Desktop\fineas\data\meta\runs\run=20260309T000000Z\features_report.json
[fineas] summary: ok_parts=2644/2772, qc_failed=0, error=0, missing_norm=128

features_report.json -> FOUND
{
  "summary": {
    "run_id": "run=20260309T000000Z",
    "created_utc": "2026-03-10T04:18:10.714909+00:00",
    "interval": "1d",
    "start": "2018-01-01",
    "end": "2026-03-10",
    "symbols": [
      "AAPL",
      "MSFT",
      "NVDA",
      "AMZN",
      "GOOGL",
      "META",
      "BRK-B",
      "JPM",
      "V",
      "WMT",
      "AMD",
      "CRM",
      "UBER",
      "CRWD",
      "SNOW",
      "SOFI",
      "BRK-A",
      "CELH",
      "^DJI",
      "^IXIC",
      "BTC-USD",
      "ETH-USD",
      "SOL-USD",
      "BNB-USD",
      "XRP-USD",
      "ADA-USD",
      "DOGE-USD",
      "LINK-USD"
    ],
    

In [17]:
summary = features_report.get("summary", {})
print("qc_failed_parts =", summary.get("qc_failed_parts"))
print("missing_norm_parts =", summary.get("missing_norm_parts"))
print("error_parts =", summary.get("error_parts"))

if (
    summary.get("qc_failed_parts", 0) == 0
    and summary.get("missing_norm_parts", 0) == 0
    and summary.get("error_parts", 0) == 0
):
    print("Phase 3 is clean. No triage records to inspect.")
else:
    print("Phase 3 has issues. Run detailed triage below.")

qc_failed_parts = 0
missing_norm_parts = 128
error_parts = 0
Phase 3 has issues. Run detailed triage below.


In [ ]:
# QC: inspect report structure directly

results = features_report.get("results", [])
print("results_n =", len(results))

results_df = pd.DataFrame(results)
print("results_df columns =", results_df.columns.tolist())
print(results_df.head(10))

results_n = 2772
results_df columns = ['symbol', 'year', 'month', 'status', 'ok', 'rows', 'out_path', 'details', 'qc']
  symbol  year  month status    ok  rows  \
0   AAPL  2018      1     ok  True    21   
1   AAPL  2018      2     ok  True    19   
2   AAPL  2018      3     ok  True    21   
3   AAPL  2018      4     ok  True    21   
4   AAPL  2018      5     ok  True    22   
5   AAPL  2018      6     ok  True    21   
6   AAPL  2018      7     ok  True    21   
7   AAPL  2018      8     ok  True    23   
8   AAPL  2018      9     ok  True    19   
9   AAPL  2018     10     ok  True    23   

                                            out_path  \
0  C:\Users\quantbase\Desktop\fineas\data\norm\fe...   
1  C:\Users\quantbase\Desktop\fineas\data\norm\fe...   
2  C:\Users\quantbase\Desktop\fineas\data\norm\fe...   
3  C:\Users\quantbase\Desktop\fineas\data\norm\fe...   
4  C:\Users\quantbase\Desktop\fineas\data\norm\fe...   
5  C:\Users\quantbase\Desktop\fineas\data\norm\fe...   
6  C

In [20]:
status_col = None
for c in ["status", "result", "outcome"]:
    if c in results_df.columns:
        status_col = c
        break

if status_col is not None:
    print(results_df[status_col].value_counts(dropna=False))
else:
    print("No explicit status column found.")

status
ok              2644
missing_norm     128
Name: count, dtype: int64


In [21]:
summary_feat = features_report.get("summary", {})
summary_norm = norm_report.get("summary", {})

feat_missing = summary_feat.get("missing_norm_parts", 0)
feat_qc_failed = summary_feat.get("qc_failed_parts", 0)
feat_error = summary_feat.get("error_parts", 0)

norm_expected = summary_norm.get("expected_parts", 0)
norm_ok = summary_norm.get("ok_parts", 0)
norm_gap = norm_expected - norm_ok

print("feat_missing_norm_parts =", feat_missing)
print("feat_qc_failed_parts =", feat_qc_failed)
print("feat_error_parts =", feat_error)
print("norm_expected - norm_ok =", norm_gap)

if feat_qc_failed == 0 and feat_error == 0 and feat_missing == norm_gap:
    print("Phase 3 is operationally clean. Missing norm parts are expected coverage gaps, not feature-QC failures.")
else:
    print("Phase 3 needs deeper triage.")

feat_missing_norm_parts = 128
feat_qc_failed_parts = 0
feat_error_parts = 0
norm_expected - norm_ok = 128
Phase 3 is operationally clean. Missing norm parts are expected coverage gaps, not feature-QC failures.


In [24]:
# choose a same-calendar sub-basket for alpha smoke

alpha_smoke_symbols = [
    "AAPL",
    "MSFT",
    "NVDA",
    "AMD",
    "UBER",
    "SOFI",
    "^IXIC",
    "BTC-USD",
]

print(alpha_smoke_symbols)

['AAPL', 'MSFT', 'NVDA', 'AMD', 'UBER', 'SOFI', '^IXIC', 'BTC-USD']


In [26]:
# helpers to locate common norm partitions and choose a month

from fineas.features.alphas import build_alpha_partition

NORM_ROOT = REPO_ROOT / "data" / "norm" / "ohlcv" / f"interval={run_spec['interval']}"

def month_key(year: int, month: int) -> str:
    return f"{year:04d}-{month:02d}"

def symbol_month_to_path(symbol: str) -> dict[str, Path]:
    root = NORM_ROOT / f"symbol={symbol}"
    out: dict[str, Path] = {}
    for path in sorted(root.glob("year=*/month=*/part-*.parquet")):
        year = int(path.parent.parent.name.split("=")[1])
        month = int(path.parent.name.split("=")[1])
        out[month_key(year, month)] = path
    return out

symbol_month_maps = {sym: symbol_month_to_path(sym) for sym in alpha_smoke_symbols}
common_months = set.intersection(*[set(m.keys()) for m in symbol_month_maps.values()])
common_months = sorted(common_months)

# latest complete month relative to end date in run spec
end_ts = pd.Timestamp(run_spec["end"])
latest_complete_month = (end_ts.to_period("M") - 1).strftime("%Y-%m")

eligible_months = [m for m in common_months if m <= latest_complete_month]
if not eligible_months:
    raise RuntimeError("No eligible complete common month found.")

chosen_month = eligible_months[-1]
print("chosen_month =", chosen_month)

print("chosen_month =", chosen_month)
#print("chosen_prev_month =", chosen_prev_month)

chosen_month = 2026-02
chosen_month = 2026-02


In [27]:
# load current + prior tail combined frames across the sub-basket

def load_month_frame(symbols: list[str], month: str) -> pd.DataFrame:
    parts = []
    for sym in symbols:
        path = symbol_month_maps[sym][month]
        df = pd.read_parquet(path)
        parts.append(df)
    out = pd.concat(parts, axis=0, ignore_index=False)
    out = out.sort_values(["ts", "symbol"]).reset_index(drop=True)
    return out

def load_symbol_history_up_to_month(symbol: str, target_month: str) -> pd.DataFrame:
    month_map = symbol_month_maps[symbol]
    chosen = []
    for m, path in sorted(month_map.items()):
        if m < target_month:
            chosen.append(pd.read_parquet(path))
    if not chosen:
        return pd.DataFrame()
    out = pd.concat(chosen, axis=0, ignore_index=False)
    out = out.sort_values(["ts"]).reset_index(drop=True)
    return out

def build_prior_tail_by_rows(symbols: list[str], target_month: str, prior_rows: int) -> pd.DataFrame:
    tails = []
    counts = {}
    for sym in symbols:
        hist = load_symbol_history_up_to_month(sym, target_month)
        tail = hist.tail(prior_rows).copy()
        counts[sym] = len(tail)
        tails.append(tail)
    out = pd.concat(tails, axis=0, ignore_index=False)
    out = out.sort_values(["ts", "symbol"]).reset_index(drop=True)
    return out, counts

df_current = load_month_frame(alpha_smoke_symbols, chosen_month)
df_prior_tail, prior_tail_counts = build_prior_tail_by_rows(alpha_smoke_symbols, chosen_month, prior_rows=90)

print("df_current.shape =", df_current.shape)
print("df_prior_tail.shape =", df_prior_tail.shape)
print("prior_tail_counts =", prior_tail_counts)
print(df_current.head())

df_current.shape = (161, 10)
df_prior_tail.shape = (720, 10)
prior_tail_counts = {'AAPL': 90, 'MSFT': 90, 'NVDA': 90, 'AMD': 90, 'UBER': 90, 'SOFI': 90, '^IXIC': 90, 'BTC-USD': 90}
                         ts   symbol          open          high  \
0 2026-02-01 00:00:00+00:00  BTC-USD  78626.125000  79322.609375   
1 2026-02-02 00:00:00+00:00     AAPL    260.029999    270.489990   
2 2026-02-02 00:00:00+00:00      AMD    235.770004    249.970001   
3 2026-02-02 00:00:00+00:00  BTC-USD  76968.875000  79258.609375   
4 2026-02-02 00:00:00+00:00     MSFT    430.239990    430.739990   

            low         close     adj_close       volume   returns movement  
0  75698.898438  76974.445312  76974.445312  53372509744 -2.094440     fall  
1    259.209991    270.010010    269.757599     73913400  4.058122     rise  
2    235.000000    246.270004    246.270004     36308100  4.029911     rise  
3  74551.335938  78688.765625  78688.765625  75140589684  2.227129     rise  
4    422.250000    4

In [28]:
# run the end-to-end alpha builder smoke

requested_alphas = [
    "alpha_001",
    "alpha_002",
    "alpha_007",
    "alpha_013",
    "alpha_021",
    "alpha_028",
    "alpha_055",
    "alpha_047",   # intentionally deferred
]

res = build_alpha_partition(
    df_current=df_current,
    alpha_ids=requested_alphas,
    df_prior_tail=df_prior_tail,
    partition_name=f"e2e_alpha_smoke/{chosen_month}",
    skip_unsupported=True,
)

print("output_columns =", res.df_alpha.columns.tolist())
print("\nmeta =")
print(json.dumps(res.meta, indent=2, default=str)[:5000])

print("\nqc =")
print(res.qc.to_dict())

print("\noutput tail =")
print(res.df_alpha.tail(12))

output_columns = ['ts', 'symbol', 'adj_close', 'returns', 'movement', 'alpha_001', 'alpha_002', 'alpha_007', 'alpha_013', 'alpha_021', 'alpha_028', 'alpha_055']

meta =
{
  "partition_name": "e2e_alpha_smoke/2026-02",
  "requested_alpha_ids": [
    "alpha_001",
    "alpha_002",
    "alpha_007",
    "alpha_013",
    "alpha_021",
    "alpha_028",
    "alpha_055",
    "alpha_047"
  ],
  "expected_alpha_cols": null,
  "expected_alpha_cols_for_qc": [
    "alpha_001",
    "alpha_002",
    "alpha_007",
    "alpha_013",
    "alpha_021",
    "alpha_028",
    "alpha_055"
  ],
  "carry_through_cols": [
    "ts",
    "symbol",
    "adj_close",
    "returns",
    "movement"
  ],
  "input_rows_current": 161,
  "input_rows_prior_tail": 720,
  "input_rows_combined": 881,
  "used_prior_tail": true,
  "required_min_prior_rows": 66,
  "constraint_driving_alphas": [
    "alpha_007"
  ],
  "prior_rows_by_symbol": {
    "AAPL": 90,
    "AMD": 90,
    "BTC-USD": 90,
    "MSFT": 90,
    "NVDA": 90,
    "SOFI"

In [29]:
# hard checks

assert res.meta["used_prior_tail"] is True
assert res.meta["continuity_deferred"] is False
assert res.meta["qc_deferred"] is False
assert res.meta["qc_ok"] is True
assert len(res.df_alpha) == len(df_current)

computed_alphas = res.meta["subset_meta"]["computed_alpha_ids"]
skipped_alphas = res.meta["subset_meta"]["skipped_alpha_ids"]

print("computed_alphas =", computed_alphas)
print("skipped_alphas =", skipped_alphas)
print("skipped_reasons =", res.meta["subset_meta"]["skipped_reasons"])
print("required_min_prior_rows =", res.meta["required_min_prior_rows"])
print("constraint_driving_alphas =", res.meta["constraint_driving_alphas"])
print("prior_rows_by_symbol =", res.meta["prior_rows_by_symbol"])

for col in computed_alphas:
    print(
        col,
        "non_null =", int(res.df_alpha[col].notna().sum()),
        "nan_n =", int(res.df_alpha[col].isna().sum()),
    )

computed_alphas = ['alpha_001', 'alpha_002', 'alpha_007', 'alpha_013', 'alpha_021', 'alpha_028', 'alpha_055']
skipped_alphas = ['alpha_047']
skipped_reasons = {'alpha_047': "alpha_047: unsupported for current run. blocking_reasons=['runtime_spec_status=deferred_missing_input', 'input_unavailable:vwap:unsupported_current_run']"}
required_min_prior_rows = 66
constraint_driving_alphas = ['alpha_007']
prior_rows_by_symbol = {'AAPL': 90, 'AMD': 90, 'BTC-USD': 90, 'MSFT': 90, 'NVDA': 90, 'SOFI': 90, 'UBER': 90, '^IXIC': 90}
alpha_001 non_null = 161 nan_n = 0
alpha_002 non_null = 161 nan_n = 0
alpha_007 non_null = 161 nan_n = 0
alpha_013 non_null = 161 nan_n = 0
alpha_021 non_null = 161 nan_n = 0
alpha_028 non_null = 161 nan_n = 0
alpha_055 non_null = 112 nan_n = 49


In [30]:
# cross-sectional sanity view

alpha_view_cols = ["ts", "symbol"] + computed_alphas
alpha_view = res.df_alpha[alpha_view_cols].copy()

for col in ["alpha_001", "alpha_013", "alpha_055"]:
    if col in alpha_view.columns:
        print(f"\n=== {col} latest snapshot ===")
        latest_ts = alpha_view["ts"].max()
        snap = alpha_view.loc[alpha_view["ts"] == latest_ts, ["symbol", col]].sort_values(col)
        print("latest_ts =", latest_ts)
        print(snap)


=== alpha_001 latest snapshot ===
latest_ts = 2026-02-28 00:00:00+00:00
      symbol  alpha_001
160  BTC-USD       -0.5

=== alpha_013 latest snapshot ===
latest_ts = 2026-02-28 00:00:00+00:00
      symbol  alpha_013
160  BTC-USD       -0.0

=== alpha_055 latest snapshot ===
latest_ts = 2026-02-28 00:00:00+00:00
      symbol  alpha_055
160  BTC-USD   -0.53936


In [32]:
# compact run summary

summary = {
    "run_id": RUN_ID,
    "basket_name": basket_name,
    "basket_size": len(basket_symbols),
    "alpha_smoke_symbols": alpha_smoke_symbols,
    "chosen_month": chosen_month,
    # "chosen_prev_month": chosen_prev_month,
    "requested_alphas": requested_alphas,
    "computed_alphas": res.meta["subset_meta"]["computed_alpha_ids"],
    "skipped_alphas": res.meta["subset_meta"]["skipped_alpha_ids"],
    "qc_ok": res.qc.ok,
    "qc_warnings": list(res.qc.warnings),
}
print(json.dumps(summary, indent=2, default=str))

{
  "run_id": "run=20260309T000000Z",
  "basket_name": "demo_small",
  "basket_size": 28,
  "alpha_smoke_symbols": [
    "AAPL",
    "MSFT",
    "NVDA",
    "AMD",
    "UBER",
    "SOFI",
    "^IXIC",
    "BTC-USD"
  ],
  "chosen_month": "2026-02",
  "requested_alphas": [
    "alpha_001",
    "alpha_002",
    "alpha_007",
    "alpha_013",
    "alpha_021",
    "alpha_028",
    "alpha_055",
    "alpha_047"
  ],
  "computed_alphas": [
    "alpha_001",
    "alpha_002",
    "alpha_007",
    "alpha_013",
    "alpha_021",
    "alpha_028",
    "alpha_055"
  ],
  "skipped_alphas": [
    "alpha_047"
  ],
  "qc_ok": true,
  "qc_warnings": [
    "e2e_alpha_smoke/2026-02/trimmed_alpha_output: NaNs present in alpha columns (warmup likely) count=1"
  ]
}


In [38]:
#------------------------------------------

In [33]:
# final pass/fail gate

from __future__ import annotations

import math
import numpy as np
import pandas as pd

# Assumes `res` is the AlphaPartitionBuildResult from the alpha smoke cell
# and `df_current` is the target output-month frame used to build it.

def assert_alpha_smoke_pass(res, df_current) -> dict:
    failures: list[str] = []

    meta = res.meta
    qc = res.qc
    df_alpha = res.df_alpha

    # 1) continuity + QC flags
    if meta.get("used_prior_tail") is not True:
        failures.append("used_prior_tail != True")
    if meta.get("continuity_deferred") is not False:
        failures.append("continuity_deferred != False")
    if meta.get("qc_deferred") is not False:
        failures.append("qc_deferred != False")
    if meta.get("qc_ok") is not True:
        failures.append("qc_ok != True")

    # 2) QC object present and passing
    if qc is None:
        failures.append("qc is None")
    else:
        if qc.ok is not True:
            failures.append("qc.ok != True")

    # 3) output row count / index alignment
    if len(df_alpha) != len(df_current):
        failures.append(f"row_count_mismatch: df_alpha={len(df_alpha)} vs df_current={len(df_current)}")
    if not df_alpha.index.equals(df_current.index):
        failures.append("df_alpha.index != df_current.index")

    # 4) prior-row threshold metadata
    required_min_prior_rows = meta.get("required_min_prior_rows")
    prior_rows_by_symbol = meta.get("prior_rows_by_symbol", {})
    if required_min_prior_rows is None:
        failures.append("required_min_prior_rows missing from meta")
        required_min_prior_rows = 0

    insufficient_symbols = {
        sym: n for sym, n in prior_rows_by_symbol.items()
        if n < required_min_prior_rows
    }
    if insufficient_symbols:
        failures.append(f"insufficient_prior_rows: {insufficient_symbols}")

    # 5) computed vs skipped alpha bookkeeping
    subset_meta = meta.get("subset_meta", {})
    computed_alpha_ids = subset_meta.get("computed_alpha_ids", [])
    skipped_alpha_ids = subset_meta.get("skipped_alpha_ids", [])
    skipped_reasons = subset_meta.get("skipped_reasons", {})

    if not computed_alpha_ids:
        failures.append("computed_alpha_ids is empty")

    # Optional but expected in this smoke:
    if "alpha_047" in skipped_alpha_ids:
        reason = skipped_reasons.get("alpha_047", "")
        if "unsupported_current_run" not in reason and "deferred_missing_input" not in reason:
            failures.append("alpha_047 skipped but reason does not show expected unsupported/deferred cause")

    # 6) alpha columns exist and contain no inf
    missing_alpha_cols = [c for c in computed_alpha_ids if c not in df_alpha.columns]
    if missing_alpha_cols:
        failures.append(f"missing_computed_alpha_columns: {missing_alpha_cols}")

    inf_counts: dict[str, int] = {}
    nan_counts: dict[str, int] = {}
    all_nan_cols: list[str] = []

    for col in computed_alpha_ids:
        if col not in df_alpha.columns:
            continue
        vals = pd.to_numeric(df_alpha[col], errors="coerce").to_numpy(dtype="float64", na_value=np.nan)
        inf_n = int(np.isinf(vals).sum())
        nan_n = int(np.isnan(vals).sum())
        inf_counts[col] = inf_n
        nan_counts[col] = nan_n
        if inf_n > 0:
            failures.append(f"inf_detected:{col}={inf_n}")
        if nan_n == len(vals):
            all_nan_cols.append(col)

    if all_nan_cols:
        failures.append(f"all_nan_alpha_columns: {all_nan_cols}")

    # 7) carry-through schema
    required_carry = ["ts", "symbol", "adj_close", "returns", "movement"]
    missing_carry = [c for c in required_carry if c not in df_alpha.columns]
    if missing_carry:
        failures.append(f"missing_carry_through_cols: {missing_carry}")

    passed = len(failures) == 0

    summary = {
        "passed": passed,
        "failures": failures,
        "required_min_prior_rows": required_min_prior_rows,
        "constraint_driving_alphas": meta.get("constraint_driving_alphas", []),
        "prior_rows_by_symbol": prior_rows_by_symbol,
        "computed_alpha_ids": computed_alpha_ids,
        "skipped_alpha_ids": skipped_alpha_ids,
        "skipped_reasons": skipped_reasons,
        "qc_ok": None if qc is None else qc.ok,
        "qc_warnings": [] if qc is None else list(qc.warnings),
        "inf_counts": inf_counts,
        "nan_counts": nan_counts,
        "row_count_df_current": int(len(df_current)),
        "row_count_df_alpha": int(len(df_alpha)),
    }
    return summary


final_check = assert_alpha_smoke_pass(res, df_current)

print("FINAL PASS =", final_check["passed"])
print("\nFAILURES:")
for x in final_check["failures"]:
    print("-", x)

print("\nSUMMARY:")
for k in [
    "required_min_prior_rows",
    "constraint_driving_alphas",
    "prior_rows_by_symbol",
    "computed_alpha_ids",
    "skipped_alpha_ids",
    "qc_ok",
    "row_count_df_current",
    "row_count_df_alpha",
]:
    print(f"{k} =", final_check[k])

FINAL PASS = True

FAILURES:

SUMMARY:
required_min_prior_rows = 66
constraint_driving_alphas = ['alpha_007']
prior_rows_by_symbol = {'AAPL': 90, 'AMD': 90, 'BTC-USD': 90, 'MSFT': 90, 'NVDA': 90, 'SOFI': 90, 'UBER': 90, '^IXIC': 90}
computed_alpha_ids = ['alpha_001', 'alpha_002', 'alpha_007', 'alpha_013', 'alpha_021', 'alpha_028', 'alpha_055']
skipped_alpha_ids = ['alpha_047']
qc_ok = True
row_count_df_current = 161
row_count_df_alpha = 161


In [34]:
# compact diagnostic printout

import json

print(json.dumps(final_check, indent=2, default=str)[:8000])

{
  "passed": true,
  "failures": [],
  "required_min_prior_rows": 66,
  "constraint_driving_alphas": [
    "alpha_007"
  ],
  "prior_rows_by_symbol": {
    "AAPL": 90,
    "AMD": 90,
    "BTC-USD": 90,
    "MSFT": 90,
    "NVDA": 90,
    "SOFI": 90,
    "UBER": 90,
    "^IXIC": 90
  },
  "computed_alpha_ids": [
    "alpha_001",
    "alpha_002",
    "alpha_007",
    "alpha_013",
    "alpha_021",
    "alpha_028",
    "alpha_055"
  ],
  "skipped_alpha_ids": [
    "alpha_047"
  ],
  "skipped_reasons": {
    "alpha_047": "alpha_047: unsupported for current run. blocking_reasons=['runtime_spec_status=deferred_missing_input', 'input_unavailable:vwap:unsupported_current_run']"
  },
  "qc_ok": true,
  "qc_warnings": [
    "e2e_alpha_smoke/2026-02/trimmed_alpha_output: NaNs present in alpha columns (warmup likely) count=1"
  ],
  "inf_counts": {
    "alpha_001": 0,
    "alpha_002": 0,
    "alpha_007": 0,
    "alpha_013": 0,
    "alpha_021": 0,
    "alpha_028": 0,
    "alpha_055": 0
  },
  "nan_

In [35]:
assert final_check["passed"] is True, json.dumps(final_check, indent=2, default=str)[:8000]

In [ ]:
# useful readable alpha summary

computed_alpha_ids = final_check["computed_alpha_ids"]

summary_rows = []
for col in computed_alpha_ids:
    s = pd.to_numeric(res.df_alpha[col], errors="coerce")
    summary_rows.append(
        {
            "alpha": col,
            "non_null": int(s.notna().sum()),
            "nan_n": int(s.isna().sum()),
            "min": None if s.notna().sum() == 0 else float(s.min()),
            "max": None if s.notna().sum() == 0 else float(s.max()),
            "mean": None if s.notna().sum() == 0 else float(s.mean()),
        }
    )

alpha_summary_df = pd.DataFrame(summary_rows).sort_values("alpha").reset_index(drop=True)
print(alpha_summary_df)

       alpha  non_null  nan_n       min       max      mean
0  alpha_001       161      0 -0.500000  0.500000 -0.027950
1  alpha_002       161      0 -0.919239  0.948276  0.043400
2  alpha_007       161      0 -1.000000 -1.000000 -1.000000
3  alpha_013       161      0 -1.000000 -0.000000 -0.472050
4  alpha_021       161      0 -1.000000  1.000000 -0.478261
5  alpha_028       161      0 -1.000000  1.000000  0.034351
6  alpha_055       112     49 -0.981981  0.934199 -0.050834


In [39]:
# note: 49 NaNs is acceptable for the smoke because:

#QC passed
#it is not all-NaN
#no inf values exist
#the warning count is only 1

#So this is not a blocker.
#But it is still worth understanding later, because alpha_055 is the only computed alpha carrying visible sparsity at the output month level.

In [37]:
# latest cross-sectional snapshot

latest_ts = res.df_alpha["ts"].max()
snapshot_cols = ["symbol"] + final_check["computed_alpha_ids"]
latest_snapshot = (
    res.df_alpha.loc[res.df_alpha["ts"] == latest_ts, snapshot_cols]
    .sort_values("symbol")
    .reset_index(drop=True)
)

print("latest_ts =", latest_ts)
print(latest_snapshot)

latest_ts = 2026-02-28 00:00:00+00:00
    symbol  alpha_001  alpha_002  alpha_007  alpha_013  alpha_021  alpha_028  \
0  BTC-USD       -0.5   -0.04609       -1.0       -0.0       -1.0       -1.0   

   alpha_055  
0   -0.53936  
